# Stage 2 — retrain CatBoost on Batch 1 + Batch 2

This notebook consolidates Batch 2 ground truth and joins it to Batch 1 past loans to form 200,000 labeled rows. It tunes **CatBoost only** using the Stage 1 5-fold CV log-loss recipe, then refits on the full 200k and predicts default probabilities for Batch 2 (reference pool) and Batch 3 (Stage 2 targets).

Batch 2 applicant-level Stage 1 CatBoost PDs are included in `Batch2.csv` for traceability; they are not used as model inputs. The outcomes are joined on unique `id`. `sex` is excluded to stay consistent with the lender's feature policy; `employment` is a native CatBoost categorical feature. This is a supervised retrain: Batch 2 defaults are observed labels, and Batch 3 labels are never loaded.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import optuna
from catboost import CatBoostClassifier, Pool, cv as catboost_cv
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from sklearn.model_selection import StratifiedKFold

SEED = 42
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TARGET = 'default'
FEATURES = ['employment', 'married', 'income', 'signal3']
CAT_FEATURES = ['employment']
CV_FOLDS = 5
N_TRIALS = 40  # Stage 1 Optuna search budget
CB_BASE = {'loss_function': 'Logloss', 'bootstrap_type': 'Bernoulli', 'random_seed': SEED,
           'logging_level': 'Silent', 'allow_writing_files': False}
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('Only CatBoost is trained; the final fit uses all 200,000 labeled rows.')

Only CatBoost is trained; the final fit uses all 200,000 labeled rows.


## 1. Consolidate Batch 2 ground truth

Merge Batch 2 application features with the diagnostic outcomes and the Stage 1 CatBoost PD reference on unique `id`. Validate one-to-one joins before saving the combined Batch 2 dataset.

In [2]:
batch2_apps = pd.read_csv(RAW_DIR / 'NewApplications_Lender3_Round1.csv')
diag = pd.read_csv(RAW_DIR / 'Round1_Diagnostic_33.csv')
stage1_pd = pd.read_csv(PROCESSED_DIR / 'signal3_new_applications_predictions_catboost.csv')

for name, frame in [('Batch 2 applications', batch2_apps), ('Round 1 diagnostic', diag), ('Stage 1 CatBoost PDs', stage1_pd)]:
    assert frame['id'].is_unique, f'{name} has duplicate IDs'
    assert len(frame) == 100_000, f'{name} expected 100k rows, got {len(frame)}'

batch2 = batch2_apps.merge(diag, on='id', how='inner', validate='one_to_one')
batch2 = batch2.merge(stage1_pd[['id', 'pd']].rename(columns={'pd':'stage1_pd'}),
                      on='id', how='inner', validate='one_to_one')
assert len(batch2) == 100_000 and batch2['default'].notna().all()
assert batch2['default'].isin([0, 1]).all()
batch2.to_csv(PROCESSED_DIR / 'Batch2.csv', index=False)
print(f'Batch 2 consolidated: {len(batch2):,} rows; default rate {batch2.default.mean():.3%}')
batch2.head()

Batch 2 consolidated: 100,000 rows; default rate 9.948%


,id,sex,employment,married,income,signal3,own,competing1,competing2,borrowertype,accepted,default,profit,stage1_pd
0,1,M,employed,0,29758,0.000000,0.057185,0.039284,0.0559,3,1,0,571.85004,0.026549
1,2,F,retired,1,6140,0.000000,0.223785,NaN,NaN,1,1,1,-10000.00000,0.113811
2,3,F,employed,1,27186,0.000000,0.054445,0.043756,0.0739,3,1,0,544.45001,0.025030
3,4,M,employed,1,8429,0.734011,0.153655,0.115800,NaN,3,0,0,NaN,0.078289
4,5,M,retired,1,2820,0.000000,0.297273,0.159800,NaN,1,0,0,NaN,0.149204


## 2. Assemble the 200,000-row supervised training set

Batch 1 comes from `PastLoans.csv`; Batch 2 adds applicant features and observed defaults. Keep the target and CatBoost feature schema aligned. The source files do not carry a Batch 1 applicant ID, so a source-local sequential ID is added only for auditability; it is not a model input.

In [3]:
batch1 = pd.read_csv(RAW_DIR / 'PastLoans.csv')
assert len(batch1) == 100_000
assert set(FEATURES + [TARGET]).issubset(batch1.columns)
assert set(FEATURES + [TARGET]).issubset(batch2.columns)

batch1_model = batch1[FEATURES + [TARGET]].copy()
batch1_model.insert(0, 'batch', 'Batch1')
batch2_model = batch2[FEATURES + [TARGET]].copy()
batch2_model.insert(0, 'batch', 'Batch2')
combined = pd.concat([batch1_model, batch2_model], ignore_index=True)
combined.insert(0, 'training_row_id', np.arange(1, len(combined) + 1))
assert len(combined) == 200_000
assert combined[TARGET].notna().all() and combined[TARGET].isin([0, 1]).all()
combined.to_csv(PROCESSED_DIR / 'Batch1_2_Combined.csv', index=False)
print(combined.groupby('batch').agg(n=('default','size'), default_rate=('default','mean')))
print('Saved Batch1_2_Combined.csv:', combined.shape)

             n  default_rate
batch                       
Batch1  100000       0.09947
Batch2  100000       0.09948
Saved Batch1_2_Combined.csv: (200000, 7)


## 3. Prepare features and tune CatBoost

Use the Stage 1 feature contract: `employment`, `married`, `income`, `signal3`, with `employment` categorical. Hyperparameter search minimizes stratified 5-fold CV log loss, and early stopping selects iteration count. Batch 3 is not used during tuning.

In [ ]:
X = combined[FEATURES].copy()
X['employment'] = X['employment'].astype(str)
y = combined[TARGET].astype(int)
train_pool = Pool(X, label=y, cat_features=CAT_FEATURES)

def objective(trial):
    params = CB_BASE | {
        'iterations': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 2, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 30, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
    }
    cv = catboost_cv(train_pool, params, fold_count=CV_FOLDS, partition_random_seed=SEED,
                     early_stopping_rounds=50, verbose=False)
    best_idx = cv['test-Logloss-mean'].idxmin()
    trial.set_user_attr('n_rounds', int(cv.loc[best_idx, 'iterations']) + 1)
    trial.set_user_attr('cv_std', float(cv.loc[best_idx, 'test-Logloss-std']))
    return float(cv.loc[best_idx, 'test-Logloss-mean'])

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS)
best_params = CB_BASE | study.best_params
best_rounds = int(study.best_trial.user_attrs['n_rounds'])
print(f'Best CV log loss={study.best_value:.5f}; rounds={best_rounds}; params={study.best_params}')

## 4. Refit on all 200,000 rows; generate reference and target PDs

The final CatBoost model is fit on the entire labeled combined set. Batch 2 reference PDs are recomputed with this model (rather than reusing Stage 1 scores) to keep local risk estimates consistent with the retrained model. Batch 3 remains unlabeled and is used only for prediction.

In [ ]:
model = CatBoostClassifier(**best_params, iterations=best_rounds)
model.fit(X, y, cat_features=CAT_FEATURES, verbose=False)

batch2_X = batch2[FEATURES].copy()
batch2_X['employment'] = batch2_X['employment'].astype(str)
batch3 = pd.read_csv(RAW_DIR / 'NewApplications_Lender3_Round2.csv')
assert batch3['id'].is_unique and len(batch3) == 100_000
assert set(FEATURES).issubset(batch3.columns)
batch3_X = batch3[FEATURES].copy()
batch3_X['employment'] = batch3_X['employment'].astype(str)

p_batch2 = model.predict_proba(batch2_X)[:, 1]
p_batch3 = model.predict_proba(batch3_X)[:, 1]
batch2_pd = pd.DataFrame({'id': batch2['id'].to_numpy(), 'pd': p_batch2})
batch3_pd = pd.DataFrame({'id': batch3['id'].to_numpy(), 'pd': p_batch3})
assert batch2_pd['id'].is_unique and batch3_pd['id'].is_unique
assert batch2_pd['pd'].between(0, 1).all() and batch3_pd['pd'].between(0, 1).all()
batch2_pd.to_csv(PROCESSED_DIR / 'batch2_p_default_catboost.csv', index=False)
batch3_pd.to_csv(PROCESSED_DIR / 'batch3_p_default_catboost.csv', index=False)

metrics = {'cv_logloss': float(study.best_value), 'cv_logloss_std': float(study.best_trial.user_attrs['cv_std']),
           'n_rounds': best_rounds, 'best_params': study.best_params,
           'training_rows': len(combined), 'batch2_pd_mean': float(p_batch2.mean()),
           'batch3_pd_mean': float(p_batch3.mean())}
(PROCESSED_DIR / 'stage2_catboost_model_metadata.json').write_text(json.dumps(metrics, indent=2))
print('Batch 2 PD mean:', round(float(p_batch2.mean()), 5), '| observed default:', round(float(batch2.default.mean()), 5))
print('Batch 3 rows:', len(batch3_pd), '| PD range:', (float(p_batch3.min()), float(p_batch3.max())))
print('Saved batch2_p_default_catboost.csv, batch3_p_default_catboost.csv, and model metadata.')